# 3 · Data preparation

| | |
|---|---|
| Lifecycle stage | 3 of 8: *Data preparation* (clean, transform, validate) |
| Tools | pandas, Dask, PyArrow (Parquet) |
| Code | `validate.py` (data contract), `clean.py` (silver), `quality.py` (audit), `features.py`, `split.py`, `prepare.py` |
| Outputs | `data/silver/transactions.parquet`, `data/gold/features_{train,val,test}.parquet`, `reports/validation_report.json`, `reports/cleaning_audit.json` |

```
bronze CSVs ──► data contract ──► silver (typed, de-duplicated, PII removed, card id hashed)
            ──► features (only each card's PRIOR transactions) ──► chronological split ──► gold
```

In [1]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path(os.environ.get("FRAUDLENS_ROOT") or (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()
CODE = Path(os.environ.get("FRAUDLENS_CODE", ROOT)).resolve()
sys.path.insert(0, str(CODE / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from fraudlens import viz
from fraudlens.utils import load_config, read_json, read_table

viz.setup()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
cfg = load_config(root=ROOT)
P = cfg["paths"]
print(f"Project root: {ROOT.name} | data present: {(P['silver'] / 'transactions.parquet').exists()}")
from fraudlens import features as F
from fraudlens.clean import SILVER_COLUMNS, hash_card_numbers
from fraudlens.datasets import load_features, load_silver

Project root: fraudlens | data present: True


## 3.1 Validate: the data contract

Before anything is transformed, each raw file is checked against a contract (`validate.py`): required
columns, types, value ranges, nulls, known categories, unique transaction ids. Failures stop the pipeline;
warnings are recorded and handled in cleaning.

In [2]:
val = read_json(P["reports"] / "validation_report.json")
for rep in val:
    print(f"{rep['dataset']}: {rep['status']}  rows={rep['profile']['rows']:,}  fraud rate={rep['profile']['fraud_rate_pct']}%")
pd.DataFrame([{**c, "dataset": rep["dataset"]} for rep in val for c in rep["checks"]])[["dataset", "check", "status", "detail"]]

fraudTrain.csv: WARN  rows=1,296,675  fraud rate=0.5789%
fraudTest.csv: WARN  rows=555,719  fraud rate=0.386%


,dataset,check,status,detail
0,fraudTrain.csv,required_columns_present,PASS,{'missing': []}
1,fraudTrain.csv,no_unexpected_columns,PASS,{'extra': []}
2,fraudTrain.csv,critical_columns_not_null,PASS,{}
3,fraudTrain.csv,timestamps_parseable,PASS,"{'unparseable': 0, 'min': '2019-01-01T00:00:18..."
4,fraudTrain.csv,dob_parseable_and_before_txn,PASS,{'unparseable': 0}
5,fraudTrain.csv,amount_positive,PASS,"{'non_positive': 0, 'min': 1.0, 'max': 28948.9}"
6,fraudTrain.csv,coordinates_in_range,PASS,{'out_of_range_rows': 0}
7,fraudTrain.csv,label_is_binary,PASS,"{'values': [0, 1]}"
8,fraudTrain.csv,gender_values_known,PASS,"{'values': ['F', 'M']}"
9,fraudTrain.csv,categories_known,PASS,{'unknown': []}


## 3.2 Clean: the silver layer

* drop the leftover index column, parse timestamps and dates, cast the label to `int8`;
* remove exact duplicate transaction ids and impossible rows (none were found in this dataset);
* strip the generator's `fraud_` prefix from every merchant name (an artefact, not a label);
* **privacy:** drop `first`, `last`, `street`; replace the card number with a salted SHA-256 `card_id`.

In [3]:
run = read_json(P["reports"] / "pipeline_run.json")
display(pd.DataFrame(run["cleaning"]))
demo = pd.Series([4263982640269299, 4263982640269299, 30074693890476])
print("hash_card_numbers ->", hash_card_numbers(demo, salt="demo-salt").tolist(), "(same card -> same id, not reversible)")
silver = load_silver(cfg)
print(f"silver: {len(silver):,} rows x {silver.shape[1]} columns, {silver.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory")
assert not {"first", "last", "street", "cc_num"} & set(silver.columns)
silver.dtypes.to_frame("dtype").T

,source_split,rows_in,duplicates_removed,invalid_rows_removed,rows_out
0,train,1296675,0,0,1296675
1,test,555719,0,0,555719


hash_card_numbers -> ['b18e58369fa17106', 'b18e58369fa17106', '357f183bbaa64f63'] (same card -> same id, not reversible)


silver: 1,852,394 rows x 19 columns, 272 MB in memory


,transaction_id,ts,card_id,merchant,category,amount,gender,dob,job,city,state,zip,city_pop,lat,lon,merch_lat,merch_lon,is_fraud,source_split
dtype,str,datetime64[us],str,category,category,float64,category,datetime64[us],category,category,category,int64,int64,float64,float64,float64,float64,int8,category


## 3.3 Data-quality audit and cleaning decisions

`quality.py` profiles the silver data and records every cleaning decision with its reason. The key one:
**amount outliers are kept**, because three quarters of all fraud sits in the top 5% of amounts; removing
"outliers" would remove the signal. Scale-sensitive models get train-fitted clipping instead.

In [4]:
audit = read_json(P["reports"] / "cleaning_audit.json")
display(pd.DataFrame(audit["outliers_train"]).T)
display(pd.Series(audit["consistency"]).to_frame("value").T)
pd.DataFrame(audit["decisions"])

,outlier_rows,outlier_pct,fraud_rate_in_outliers_pct,fraud_rate_overall_pct,share_of_all_fraud_pct
amount,"61,564.0000",5.1900,8.4370,0.5770,75.9700
city_pop,"222,042.0000",18.7400,0.6120,0.5770,19.8600
age,743.0000,0.0600,0.0000,0.5770,0.0000
distance_km,0.0000,0.0000,NaN,0.5770,0.0000


,merchants,merchants_with_multiple_categories,cards,cards_with_conflicting_profile,age_min,age_max,ages_below_16,ages_above_100
value,693.0000,7.0000,999.0000,0.0000,13.9000,96.2000,"12,184.0000",0.0000


,issue,decision,reason
0,Extreme transaction amounts (IQR outliers),Keep all rows. Use log-scaling and train-fitte...,Fraud is concentrated among high amounts; remo...
1,"Near-duplicate transactions (same card, mercha...",Keep and report. They have distinct transactio...,Rapid repeats are a known fraud pattern; the v...
2,Missing history features on a card's first tra...,Keep NaN in the tree view; median-impute with ...,The absence of history is informative and must...
3,Skewed and heavy-tailed numeric features,No change to stored data; handled in preproces...,Cleaning keeps values faithful; model-specific...
4,"High-cardinality text columns (merchant, job, ...",Excluded from the model matrices for now.,"They need label-based encoding, which must be ..."


## 3.4 Scaling out with Dask

The raw files fit in memory on a laptop, but the same preparation has to work when they do not. Dask
reads the CSVs in parallel 64 MB partitions with the pandas API. Here it recomputes monthly volume and
fraud counts straight from the raw files, and the result must match the pandas silver layer exactly
(no rows were removed). At production scale the same logic would run on a Dask cluster or PySpark.

In [5]:
import time
import dask
import dask.dataframe as dd

raw_files = [str(P["raw"] / f) for f in cfg["dataset"]["files"].values()]
t0 = time.perf_counter()
ddf = dd.read_csv(raw_files, usecols=["trans_date_trans_time", "amt", "is_fraud"], blocksize="64MB")
ddf = ddf.assign(month=ddf["trans_date_trans_time"].str.slice(0, 7))
monthly_dask = ddf.groupby("month").agg({"is_fraud": ["count", "sum"], "amt": "sum"}).compute()
monthly_dask.columns = ["transactions", "frauds", "amount"]
monthly_dask = monthly_dask.sort_index()
print(f"Dask: {ddf.npartitions} partitions, {time.perf_counter() - t0:.1f}s, dask {dask.__version__}")

monthly_pd = (silver.assign(month=silver["ts"].dt.strftime("%Y-%m"), is_fraud=silver["is_fraud"].astype("int64")).groupby("month")
              .agg(transactions=("is_fraud", "size"), frauds=("is_fraud", "sum"), amount=("amount", "sum")))
same = (monthly_dask[["transactions", "frauds"]].astype(int).equals(monthly_pd[["transactions", "frauds"]].astype(int))
        and np.allclose(monthly_dask["amount"], monthly_pd["amount"]))
print("Dask (raw CSV) and pandas (silver) agree:", same)
monthly_pd.assign(fraud_rate_pct=lambda t: 100 * t["frauds"].astype(float) / t["transactions"]).head(24)

Dask: 7 partitions, 4.9s, dask 2026.8.0


Dask (raw CSV) and pandas (silver) agree: True


,transactions,frauds,amount,fraud_rate_pct
month,,,,
2019-01,52525,506,"3,759,750.0300",0.9634
2019-02,49866,517,"3,604,662.2300",1.0368
2019-03,70939,494,"5,027,886.4000",0.6964
2019-04,68078,376,"4,761,504.5800",0.5523
2019-05,72532,408,"5,060,903.3200",0.5625
2019-06,86064,354,"6,036,397.4200",0.4113
2019-07,86596,331,"6,044,026.7400",0.3822
2019-08,87359,382,"6,047,288.6500",0.4373
2019-09,70652,418,"4,949,834.4200",0.5916


## 3.5 Transform: leakage-safe features

Every behavioural feature is computed from the card's **previous** transactions only, exactly as a live
system would see them at authorisation time:

| Group | Features | Hypothesis (stage 1) |
|---|---|---|
| Transaction | `amount`, `log_amount`, `hour`, `day_of_week`, `is_weekend`, `is_night`, `category` | H1, H2, H4 |
| Customer | `age`, `gender`, `log_city_pop` | H5 |
| Card history | `amt_to_card_mean`, `amt_zscore_card`, `secs_since_last_txn`, `category_share_card` | H1, H4 |
| Velocity (1 h / 24 h / 7 d) | `txn_count_*`, `amt_sum_*` | H3 |

Running totals (`card_prior_txn_count`, ...) only ever grow, so later periods contain values never seen
in training; they were measured as drifting and excluded (`preprocess.exclude_features`).

In [6]:
train = load_features(cfg, "train")
num = F.numeric_features(cfg["features"]["velocity_windows"])
print(f"gold features (train): {len(train):,} rows, {train.shape[1]} columns")
train[["transaction_id", "ts", "card_id", "amount", "category", "amt_to_card_mean", "secs_since_last_txn",
       "txn_count_1h", "amt_sum_24h", "txn_count_7d", "is_fraud"]].head()

gold features (train): 1,185,094 rows, 27 columns


,transaction_id,ts,card_id,amount,category,amt_to_card_mean,secs_since_last_txn,txn_count_1h,amt_sum_24h,txn_count_7d,is_fraud
0,0b242abb623afc578575680df30655b9,2019-01-01 00:00:18,6260e63a3bbed4c9,4.9700,misc_net,NaN,NaN,0,0.0000,0,0
1,1f76529f8574734946361c461b024d99,2019-01-01 00:00:44,e315d012cc5cfb8b,107.2300,grocery_pos,NaN,NaN,0,0.0000,0,0
2,a1a22d70485983eac12b5b88dad1cf95,2019-01-01 00:00:51,b9f9c103eac8b25f,220.1100,entertainment,NaN,NaN,0,0.0000,0,0
3,6b849c168bdad6f867558c3793159a81,2019-01-01 00:01:16,608ff4d71e540f45,45.0000,gas_transport,NaN,NaN,0,0.0000,0,0
4,a41d7549acf90789359a9aa5346dcb46,2019-01-01 00:03:06,03bcfce884341f84,41.9600,misc_pos,NaN,NaN,0,0.0000,0,0


### Proof that the velocity features use only the past

Recompute `txn_count_24h` and `amt_sum_24h` for one card by brute force (a loop over the card's earlier
transactions) and compare with the vectorised pipeline values.

In [7]:
card = train["card_id"].value_counts().index[0]
c = train[train["card_id"] == card].sort_values(["ts", "transaction_id"]).head(400).reset_index(drop=True)
t = ((c["ts"] - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)).to_numpy()
brute_n = [int(((t[:j] >= t[j] - 86400)).sum()) for j in range(len(c))]
brute_s = [float(c["amount"].to_numpy()[:j][t[:j] >= t[j] - 86400].sum()) for j in range(len(c))]
print("txn_count_24h matches:", np.array_equal(brute_n, c["txn_count_24h"].to_numpy()))
print("amt_sum_24h matches:  ", np.allclose(brute_s, c["amt_sum_24h"].to_numpy()))

txn_count_24h matches:

 True
amt_sum_24h matches:   True


## 3.6 Split by time, never at random

Kaggle's `fraudTest.csv` continues `fraudTrain.csv` in time, so it becomes the **test** period. The last
six weeks of `fraudTrain.csv` become the **validation** period, used to pick the champion and its
threshold. A random split would let the model learn from the future.

In [8]:
sp = pd.DataFrame(run["split"]["splits"]).T
display(sp)
fig, ax = plt.subplots(figsize=(10, 2.4))
for i, (name, r) in enumerate(sp.iterrows()):
    start, end = pd.Timestamp(r["start"]), pd.Timestamp(r["end"])
    ax.barh(0, (end - start).days, left=start, color=viz.SERIES[i], label=f"{name} ({int(r['rows']):,} rows)")
ax.set_yticks([]); ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.45)); ax.set_title("Chronological split")
plt.show()

,rows,start,end,fraud_count,fraud_rate_pct,cards
train,1185094,2019-01-01T00:00:18,2020-05-10T12:13:21,6837,0.5769,978
val,111581,2020-05-10T12:13:54,2020-06-21T12:13:37,669,0.5996,914
test,555719,2020-06-21T12:14:25,2020-12-31T23:59:34,2145,0.3860,924


In [9]:
v = read_json(P["reports"] / "verification_report.json")
pd.DataFrame([c for c in v["checks"] if c["stage"] == "3-preparation"])[["check", "status", "detail"]]

,check,status,detail
0,raw files passed the data contract,PASS,"[WARN, WARN]"
1,silver rows = manifest rows - removed rows,PASS,"{'manifest': 1852394, 'removed': 0, 'silver': ..."
2,no direct identifiers in silver,PASS,None
3,split rows sum to silver rows,PASS,"{'train': 1185094, 'val': 111581, 'test': 555719}"
4,no transaction appears in two splits,PASS,None
5,splits are chronological (train < val < test),PASS,None
6,no feature is derived from the label,PASS,None
7,online store holds every card and every transa...,PASS,"{'cards': 999, 'transactions': 1852394}"


## 3.7 Summary

* The data contract passes (two warnings handled in cleaning); no duplicates or impossible rows.
* Silver is typed, privacy-reduced (no names, streets or card numbers) and stored as Parquet.
* Features are provably leakage-safe; the split is chronological; verification checks prove both.
* Dask reproduces the same aggregates from the raw files, so the preparation scales beyond memory.

**Next:** [04 · Exploratory data analysis](04_eda.ipynb)